In [1]:
import pandas as pd
import numpy as np

df_final = pd.read_csv("data/khonkaen_pm25_features_final.csv")
df_final["datetime_utc"] = pd.to_datetime(df_final["datetime_utc"])

print(df_final.shape)
df_final.head()

(37909, 46)


,datetime_utc,pm25,temp,humidity,wind_speed,precipitation,hour,month,dayofweek,hour_sin,...,pm25_t+15,pm25_t+16,pm25_t+17,pm25_t+18,pm25_t+19,pm25_t+20,pm25_t+21,pm25_t+22,pm25_t+23,pm25_t+24
0,2021-04-30 18:00:00+00:00,20.111111,25.1,91,4.7,0.0,18,4,4,-1.000000,...,20.000000,20.000000,20.000000,20.000000,20.055556,20.111111,20.166667,20.222222,20.277778,20.333333
1,2021-04-30 19:00:00+00:00,20.333333,24.9,91,5.0,0.0,19,4,4,-0.965926,...,20.000000,20.000000,20.000000,20.055556,20.111111,20.166667,20.222222,20.277778,20.333333,20.388889
2,2021-04-30 20:00:00+00:00,20.555556,24.5,93,4.7,0.0,20,4,4,-0.866025,...,20.000000,20.000000,20.055556,20.111111,20.166667,20.222222,20.277778,20.333333,20.388889,20.444444
3,2021-04-30 21:00:00+00:00,20.777778,24.2,95,4.0,0.0,21,4,4,-0.707107,...,20.000000,20.055556,20.111111,20.166667,20.222222,20.277778,20.333333,20.388889,20.444444,20.500000
4,2021-04-30 22:00:00+00:00,21.000000,23.9,97,4.6,0.0,22,4,4,-0.500000,...,20.055556,20.111111,20.166667,20.222222,20.277778,20.333333,20.388889,20.444444,20.500000,20.555556


In [2]:
# %% เซลล์ 2: นิยาม feature columns และ target columns

feature_cols = [
    "temp", "humidity", "wind_speed", "precipitation",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dayofweek",
    "pm25_lag_1", "pm25_lag_3", "pm25_lag_6", "pm25_lag_12", "pm25_lag_24",
    "pm25_roll_mean_6", "pm25_roll_std_6", "pm25_roll_mean_24", "pm25_roll_std_24"
]

target_cols = [f"pm25_t+{h}" for h in range(1, 25)]

print(f"จำนวน feature columns: {len(feature_cols)}")
print(f"จำนวน target columns: {len(target_cols)}")

# Train/Test split แบบ time-based (80/20)
split_idx = int(len(df_final) * 0.8)
train = df_final.iloc[:split_idx]
test = df_final.iloc[split_idx:]

X_train, y_train = train[feature_cols], train[target_cols]
X_test, y_test = test[feature_cols], test[target_cols]

print(f"\nTrain: {len(train)} แถว ({train['datetime_utc'].min()} → {train['datetime_utc'].max()})")
print(f"Test: {len(test)} แถว ({test['datetime_utc'].min()} → {test['datetime_utc'].max()})")

จำนวน feature columns: 18
จำนวน target columns: 24

Train: 30327 แถว (2021-04-30 18:00:00+00:00 → 2025-05-16 03:00:00+00:00)
Test: 7582 แถว (2025-05-16 04:00:00+00:00 → 2026-07-01 01:00:00+00:00)


In [3]:
# %% เซลล์ 3: ทดสอบเทรน LightGBM สำหรับ horizon เดียวก่อน (pm25_t+1)
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

start = time.time()

model_h1 = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
model_h1.fit(X_train, y_train["pm25_t+1"])

y_pred = model_h1.predict(X_test)

mae = mean_absolute_error(y_test["pm25_t+1"], y_pred)
rmse = mean_squared_error(y_test["pm25_t+1"], y_pred) ** 0.5
r2 = r2_score(y_test["pm25_t+1"], y_pred)

print(f"เวลาที่ใช้: {time.time() - start:.2f} วินาที")
print(f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")

เวลาที่ใช้: 2.26 วินาที
MAE: 0.445 | RMSE: 0.658 | R²: 0.998


In [4]:
# %% เซลล์ 4: ทดสอบเทรน LightGBM สำหรับ horizon ไกลสุด (pm25_t+24) เทียบกับ t+1

model_h24 = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
model_h24.fit(X_train, y_train["pm25_t+24"])

y_pred_h24 = model_h24.predict(X_test)

mae_h24 = mean_absolute_error(y_test["pm25_t+24"], y_pred_h24)
rmse_h24 = mean_squared_error(y_test["pm25_t+24"], y_pred_h24) ** 0.5
r2_h24 = r2_score(y_test["pm25_t+24"], y_pred_h24)

print(f"=== t+24 (พยากรณ์ล่วงหน้า 24 ชม.) ===")
print(f"MAE: {mae_h24:.3f} | RMSE: {rmse_h24:.3f} | R²: {r2_h24:.3f}")

print(f"\n=== เทียบกับ t+1 ===")
print(f"t+1  → MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")
print(f"t+24 → MAE: {mae_h24:.3f} | RMSE: {rmse_h24:.3f} | R²: {r2_h24:.3f}")

=== t+24 (พยากรณ์ล่วงหน้า 24 ชม.) ===
MAE: 4.251 | RMSE: 6.099 | R²: 0.825

=== เทียบกับ t+1 ===
t+1  → MAE: 0.445 | RMSE: 0.658 | R²: 0.998
t+24 → MAE: 4.251 | RMSE: 6.099 | R²: 0.825


In [5]:
# %% เซลล์ 5: เทรน LightGBM ครบทั้ง 24 horizons + เก็บผลลัพธ์

import time

lgb_models = {}
results_lgb = []

start_total = time.time()

for h in range(1, 25):
    target = f"pm25_t+{h}"
    
    model = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
    model.fit(X_train, y_train[target])
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test[target], y_pred)
    rmse = mean_squared_error(y_test[target], y_pred) ** 0.5
    r2 = r2_score(y_test[target], y_pred)
    
    lgb_models[h] = model
    results_lgb.append({"horizon": h, "MAE": mae, "RMSE": rmse, "R2": r2})
    
    print(f"t+{h:2d} → MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")

print(f"\nเวลาที่ใช้ทั้งหมด: {time.time() - start_total:.2f} วินาที")

df_results_lgb = pd.DataFrame(results_lgb)

t+ 1 → MAE: 0.445 | RMSE: 0.658 | R²: 0.998
t+ 2 → MAE: 0.657 | RMSE: 0.953 | R²: 0.996
t+ 3 → MAE: 0.874 | RMSE: 1.273 | R²: 0.992
t+ 4 → MAE: 1.063 | RMSE: 1.540 | R²: 0.989
t+ 5 → MAE: 1.253 | RMSE: 1.816 | R²: 0.984
t+ 6 → MAE: 1.458 | RMSE: 2.104 | R²: 0.979
t+ 7 → MAE: 1.613 | RMSE: 2.324 | R²: 0.975
t+ 8 → MAE: 1.828 | RMSE: 2.644 | R²: 0.967
t+ 9 → MAE: 2.031 | RMSE: 2.945 | R²: 0.959
t+10 → MAE: 2.173 | RMSE: 3.148 | R²: 0.953
t+11 → MAE: 2.355 | RMSE: 3.430 | R²: 0.945
t+12 → MAE: 2.506 | RMSE: 3.641 | R²: 0.938
t+13 → MAE: 2.679 | RMSE: 3.908 | R²: 0.928
t+14 → MAE: 2.806 | RMSE: 4.084 | R²: 0.922
t+15 → MAE: 2.974 | RMSE: 4.352 | R²: 0.911
t+16 → MAE: 3.143 | RMSE: 4.576 | R²: 0.902
t+17 → MAE: 3.326 | RMSE: 4.783 | R²: 0.892
t+18 → MAE: 3.443 | RMSE: 5.017 | R²: 0.882
t+19 → MAE: 3.619 | RMSE: 5.277 | R²: 0.869
t+20 → MAE: 3.718 | RMSE: 5.358 | R²: 0.865
t+21 → MAE: 3.892 | RMSE: 5.643 | R²: 0.850
t+22 → MAE: 4.015 | RMSE: 5.814 | R²: 0.841
t+23 → MAE: 4.136 | RMSE: 5.953 

In [6]:
# %% เซลล์ 6: เทรน XGBoost ครบทั้ง 24 horizons

import xgboost as xgb

xgb_models = {}
results_xgb = []

start_total = time.time()

for h in range(1, 25):
    target = f"pm25_t+{h}"
    
    model = xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
    model.fit(X_train, y_train[target])
    
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test[target], y_pred)
    rmse = mean_squared_error(y_test[target], y_pred) ** 0.5
    r2 = r2_score(y_test[target], y_pred)
    
    xgb_models[h] = model
    results_xgb.append({"horizon": h, "MAE": mae, "RMSE": rmse, "R2": r2})
    
    print(f"t+{h:2d} → MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}")

print(f"\nเวลาที่ใช้ทั้งหมด: {time.time() - start_total:.2f} วินาที")

df_results_xgb = pd.DataFrame(results_xgb)

t+ 1 → MAE: 0.495 | RMSE: 0.782 | R²: 0.997
t+ 2 → MAE: 0.717 | RMSE: 1.074 | R²: 0.995
t+ 3 → MAE: 0.943 | RMSE: 1.370 | R²: 0.991
t+ 4 → MAE: 1.138 | RMSE: 1.634 | R²: 0.987
t+ 5 → MAE: 1.327 | RMSE: 1.917 | R²: 0.983
t+ 6 → MAE: 1.549 | RMSE: 2.211 | R²: 0.977
t+ 7 → MAE: 1.758 | RMSE: 2.512 | R²: 0.970
t+ 8 → MAE: 1.966 | RMSE: 2.788 | R²: 0.963
t+ 9 → MAE: 2.183 | RMSE: 3.208 | R²: 0.952
t+10 → MAE: 2.383 | RMSE: 3.452 | R²: 0.944
t+11 → MAE: 2.545 | RMSE: 3.702 | R²: 0.936
t+12 → MAE: 2.710 | RMSE: 3.841 | R²: 0.931
t+13 → MAE: 2.884 | RMSE: 4.142 | R²: 0.919
t+14 → MAE: 2.969 | RMSE: 4.286 | R²: 0.914
t+15 → MAE: 3.189 | RMSE: 4.484 | R²: 0.905
t+16 → MAE: 3.354 | RMSE: 4.729 | R²: 0.895
t+17 → MAE: 3.640 | RMSE: 5.253 | R²: 0.870
t+18 → MAE: 3.708 | RMSE: 5.231 | R²: 0.871
t+19 → MAE: 3.867 | RMSE: 5.479 | R²: 0.859
t+20 → MAE: 4.086 | RMSE: 5.783 | R²: 0.843
t+21 → MAE: 4.351 | RMSE: 6.141 | R²: 0.823
t+22 → MAE: 4.430 | RMSE: 6.300 | R²: 0.813
t+23 → MAE: 4.507 | RMSE: 6.359 

In [7]:
# %% เซลล์ 7: เปรียบเทียบผลลัพธ์ 2 โมเดล

comparison = df_results_lgb.merge(df_results_xgb, on="horizon", suffixes=("_lgb", "_xgb"))
comparison["MAE_diff"] = comparison["MAE_xgb"] - comparison["MAE_lgb"]

print(comparison[["horizon", "MAE_lgb", "MAE_xgb", "MAE_diff", "R2_lgb", "R2_xgb"]].to_string(index=False))

print(f"\nMAE เฉลี่ยทั้ง 24 horizons:")
print(f"  LightGBM: {df_results_lgb['MAE'].mean():.3f}")
print(f"  XGBoost:  {df_results_xgb['MAE'].mean():.3f}")

 horizon  MAE_lgb  MAE_xgb  MAE_diff   R2_lgb   R2_xgb
       1 0.445151 0.495127  0.049976 0.997966 0.997123
       2 0.656663 0.716692  0.060030 0.995725 0.994574
       3 0.873845 0.943248  0.069403 0.992380 0.991167
       4 1.062728 1.137611  0.074883 0.988846 0.987446
       5 1.252822 1.326640  0.073819 0.984483 0.982710
       6 1.458486 1.548733  0.090247 0.979176 0.977014
       7 1.612547 1.757679  0.145132 0.974597 0.970309
       8 1.828312 1.965688  0.137376 0.967122 0.963449
       9 2.031263 2.183146  0.151883 0.959209 0.951598
      10 2.173294 2.383493  0.210199 0.953397 0.943958
      11 2.355032 2.544996  0.189963 0.944669 0.935532
      12 2.506101 2.710089  0.203989 0.937637 0.930603
      13 2.678960 2.884039  0.205080 0.928173 0.919304
      14 2.806044 2.968977  0.162933 0.921580 0.913601
      15 2.973722 3.189421  0.215699 0.910952 0.905443
      16 3.142750 3.353870  0.211120 0.901554 0.894857
      17 3.326417 3.639800  0.313384 0.892406 0.870229
      18 3

In [8]:
# %% เซลล์ 8: เตรียมข้อมูลสำหรับ Prophet

from prophet import Prophet

# Prophet ต้องการคอลัมน์ชื่อ ds (datetime) และ y (target)
df_prophet = df_final[["datetime_utc", "pm25", "temp", "humidity", "wind_speed", "precipitation"]].copy()
df_prophet.columns = ["ds", "y", "temp", "humidity", "wind_speed", "precipitation"]
df_prophet["ds"] = df_prophet["ds"].dt.tz_localize(None)  # Prophet ไม่รับ timezone-aware

# split เดียวกับที่ใช้ก่อนหน้า (80/20 ตามเวลา)
train_prophet = df_prophet.iloc[:split_idx]
test_prophet = df_prophet.iloc[split_idx:]

print(f"Train: {len(train_prophet)} แถว")
print(f"Test: {len(test_prophet)} แถว")

c:\Users\junta\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 30327 แถว
Test: 7582 แถว


In [9]:
# %% เซลล์ 9: เทรน Prophet (native multi-step ในตัว ไม่ต้อง loop 24 horizons)

import time

start = time.time()

model_prophet = Prophet()
model_prophet.add_regressor("temp")
model_prophet.add_regressor("humidity")
model_prophet.add_regressor("wind_speed")
model_prophet.add_regressor("precipitation")

model_prophet.fit(train_prophet)

print(f"เวลาเทรน: {time.time() - start:.2f} วินาที")

17:13:53 - cmdstanpy - INFO - Chain [1] start processing
17:13:59 - cmdstanpy - INFO - Chain [1] done processing


เวลาเทรน: 7.37 วินาที


In [10]:
# %% เซลล์ 10: พยากรณ์บน test set

# สร้าง future dataframe จาก test set (มี regressor values จริงอยู่แล้วเพราะเป็นข้อมูลย้อนหลัง)
future = test_prophet[["ds", "temp", "humidity", "wind_speed", "precipitation"]].copy()

forecast = model_prophet.predict(future)

y_true = test_prophet["y"].values
y_pred_prophet = forecast["yhat"].values

mae_prophet = mean_absolute_error(y_true, y_pred_prophet)
rmse_prophet = mean_squared_error(y_true, y_pred_prophet) ** 0.5
r2_prophet = r2_score(y_true, y_pred_prophet)

print(f"=== Prophet (พยากรณ์ต่อเนื่องทั้ง test set) ===")
print(f"MAE: {mae_prophet:.3f} | RMSE: {rmse_prophet:.3f} | R²: {r2_prophet:.3f}")

=== Prophet (พยากรณ์ต่อเนื่องทั้ง test set) ===
MAE: 7.586 | RMSE: 9.887 | R²: 0.540


In [11]:
# %% เซลล์ 15: เทรนโมเดลสุดท้าย (default params) 
# หมายเหตุ: เคยทดลอง hyperparameter tuning ด้วย Optuna (30 trials/horizon, ~24 นาที)
# ผลคือ MAE เฉลี่ยเท่ากับ default params เป๊ะ (2.511 ทั้งคู่) — สรุปว่า feature 
# engineering (lag/rolling) เป็นตัวขับเคลื่อนความแม่นยำหลัก ไม่ใช่ hyperparameter
# จึงตัดสินใจใช้ default params เพื่อความเรียบง่าย ลด overfitting risk ต่อ validation set

final_lgb_models = {}

for h in range(1, 25):
    target = f"pm25_t+{h}"
    model = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
    model.fit(X_train, y_train[target])
    final_lgb_models[h] = model

print(f"เทรนโมเดลสุดท้ายครบ {len(final_lgb_models)} horizons (default params)")

เทรนโมเดลสุดท้ายครบ 24 horizons (default params)


In [12]:
# %% เซลล์ 16: บันทึกโมเดล + metadata

import pickle
import os
from datetime import datetime as dt

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

timestamp = dt.now().strftime("%Y%m%d_%H%M%S")

model_package = {
    "models": final_lgb_models,
    "feature_cols": feature_cols,
    "target_cols": target_cols,
    "trained_at": timestamp,
    "station_id": 7747,
    "hyperparameters": "default (n_estimators=200); Optuna tuning tested and found no significant improvement",
    "test_mae_by_horizon": df_results_lgb.set_index("horizon")["MAE"].to_dict(),
    "test_r2_by_horizon": df_results_lgb.set_index("horizon")["R2"].to_dict(),
}

model_path = f"{MODEL_DIR}/lgb_pm25_multistep_{timestamp}.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model_package, f)

print(f"บันทึกโมเดลแล้วที่: {model_path}")
print(f"ขนาดไฟล์: {os.path.getsize(model_path) / 1024:.1f} KB")

บันทึกโมเดลแล้วที่: models/lgb_pm25_multistep_20260708_171412.pkl
ขนาดไฟล์: 13249.4 KB


In [13]:
# %% เซลล์ 17: ทดสอบโหลดโมเดลกลับมาใช้

with open(model_path, "rb") as f:
    loaded_package = pickle.load(f)

test_model_h1 = loaded_package["models"][1]
test_pred = test_model_h1.predict(X_test.iloc[[0]])
print(f"ทดสอบพยากรณ์ t+1 จากโมเดลที่โหลดกลับมา: {test_pred[0]:.2f} µg/m³")
print(f"ค่าจริง: {y_test['pm25_t+1'].iloc[0]:.2f} µg/m³")
print("\n✅ โมเดลบันทึก/โหลดใช้งานได้ปกติ")

ทดสอบพยากรณ์ t+1 จากโมเดลที่โหลดกลับมา: 16.66 µg/m³
ค่าจริง: 16.60 µg/m³

✅ โมเดลบันทึก/โหลดใช้งานได้ปกติ


In [14]:
# %% เซลล์ 18: บันทึกตารางสรุปผล MAE/RMSE/R² ทุก horizon

results_summary_path = f"{MODEL_DIR}/lgb_results_summary_{timestamp}.csv"
df_results_lgb.to_csv(results_summary_path, index=False)

print(f"บันทึกสรุปผลแล้วที่: {results_summary_path}")
print(f"\n=== สรุปภาพรวม ===")
print(f"MAE เฉลี่ย: {df_results_lgb['MAE'].mean():.3f}")
print(f"RMSE เฉลี่ย: {df_results_lgb['RMSE'].mean():.3f}")
print(f"R² เฉลี่ย: {df_results_lgb['R2'].mean():.3f}")
print(f"R² ที่ t+1 (สั้นสุด): {df_results_lgb.iloc[0]['R2']:.3f}")
print(f"R² ที่ t+24 (ไกลสุด): {df_results_lgb.iloc[-1]['R2']:.3f}")

บันทึกสรุปผลแล้วที่: models/lgb_results_summary_20260708_171412.csv

=== สรุปภาพรวม ===
MAE เฉลี่ย: 2.511
RMSE เฉลี่ย: 3.639
R² เฉลี่ย: 0.925
R² ที่ t+1 (สั้นสุด): 0.998
R² ที่ t+24 (ไกลสุด): 0.825
